# Wind Turbine SCADA & CMS Data — Exploratory Data Analysis (EDA)

This notebook walks through a complete EDA workflow for wind turbine SCADA and CMS data,
before any Feature Engineering or Modeling step.

**Structure**
1. Setup & Configuration
2. Single Sensor Visualization
3. Multiple Sensor Analysis
4. Time Series Analysis (Rolling Stats)
5. Seasonality
6. Autocorrelation
7. Multi-Turbine Analysis
8. CMS Analysis (Vibration / Temperature)
9. Feature Engineering Visualization
10. Pre-Modeling Checklist

> **How to use this notebook:** Edit the `CONFIG` cell in Section 1 to match your actual
> file path and column names. Every section below refers to those config variables, so
> you generally don't need to touch the plotting code itself — just point the config at
> the right columns and re-run.


## 1. Setup & Configuration

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Time series specific
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from pandas.plotting import lag_plot

# Scaling (used in Part 8/9)
from sklearn.preprocessing import StandardScaler, MinMaxScaler

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)
pd.set_option("display.max_columns", 100)

%matplotlib inline


In [ ]:
# ============================================================
# CONFIG — EDIT THIS CELL TO MATCH YOUR DATASET
# ============================================================

CONFIG = {
    # Path to your SCADA/CMS data file (csv, parquet, etc.)
    "file_path": "your_scada_data.csv",

    # Column holding the timestamp
    "timestamp_col": "Timestamp",

    # Column identifying the turbine (e.g. 'WT01', 'WT02', ...). Set to None if single turbine.
    "turbine_col": "Turbine_ID",

    # Primary sensors of interest
    "power_col": "ACTPWR",        # Active Power
    "wind_speed_col": "WindSpeed",
    "rotor_speed_col": "RotorSpeed",

    # Temperature sensors
    "gen_temp_col": "GenTemp",
    "bearing_temp_col": "BearingTemp",
    "gearbox_temp_col": "GearboxTemp",

    # CMS / vibration sensor(s)
    "vibration_col": "Vibration",

    # Rolling windows to evaluate (in number of rows — adjust to your sampling rate)
    "rolling_windows": [7, 30],       # e.g. 7-day, 30-day if daily data
    "roc_lag": 1,                     # rate-of-change lag (rows)
    "lag_list": [1, 6, 24, 144],      # for ACF / lag-feature exploration

    # Which turbine to use for single-turbine plots (Part 1-5, 7, 8) if turbine_col is set
    "focus_turbine": "WT01",
}

print("Config loaded. Edit the dict above before running the rest of the notebook.")


In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(CONFIG["file_path"])

# Parse timestamp and sort
df[CONFIG["timestamp_col"]] = pd.to_datetime(df[CONFIG["timestamp_col"]])
df = df.sort_values(CONFIG["timestamp_col"]).reset_index(drop=True)

print(df.shape)
df.head()


In [ ]:
# Build a single-turbine working frame (`ts`) indexed by time, used throughout Parts 1-5, 7, 8.
# If you don't have multiple turbines, this just becomes the whole dataset.

if CONFIG["turbine_col"] and CONFIG["turbine_col"] in df.columns:
    ts = df[df[CONFIG["turbine_col"]] == CONFIG["focus_turbine"]].copy()
else:
    ts = df.copy()

ts = ts.set_index(CONFIG["timestamp_col"]).sort_index()
ts.head()


## 2. Single Sensor Visualization (Part 1)

Covers: time series plot, histogram, box plot, density (KDE) plot.
Default sensor is Active Power (`power_col`) — change `sensor_col` below to inspect any other sensor.


In [ ]:
sensor_col = CONFIG["power_col"]  # <- change to inspect a different sensor

### 2.1 Time Series Plot ---------------------------------------------------
plt.figure()
plt.plot(ts.index, ts[sensor_col], linewidth=0.8)
plt.title(f"{sensor_col} over Time — {CONFIG.get('focus_turbine', '')}")
plt.xlabel("Timestamp")
plt.ylabel(sensor_col)
plt.tight_layout()
plt.show()


In [ ]:
### 2.2 Histogram ------------------------------------------------------------
plt.figure()
plt.hist(ts[sensor_col].dropna(), bins=50, edgecolor="black")
plt.title(f"Distribution of {sensor_col}")
plt.xlabel(sensor_col)
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
### 2.3 Box Plot --------------------------------------------------------------
plt.figure(figsize=(6, 5))
sns.boxplot(y=ts[sensor_col])
plt.title(f"Box Plot of {sensor_col}")
plt.tight_layout()
plt.show()


In [ ]:
### 2.4 Density Plot (KDE) -----------------------------------------------------
plt.figure()
sns.kdeplot(ts[sensor_col].dropna(), fill=True)
plt.title(f"Density Plot (KDE) of {sensor_col}")
plt.xlabel(sensor_col)
plt.tight_layout()
plt.show()


## 3. Multiple Sensor Analysis (Part 2)

Covers: scatter plots, pair plot, correlation heatmap.


In [ ]:
### 3.1 Scatter Plots ----------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].scatter(ts[CONFIG["wind_speed_col"]], ts[CONFIG["power_col"]], s=5, alpha=0.4)
axes[0].set_xlabel(CONFIG["wind_speed_col"]); axes[0].set_ylabel(CONFIG["power_col"])
axes[0].set_title("Wind Speed vs Power")

axes[1].scatter(ts[CONFIG["rotor_speed_col"]], ts[CONFIG["power_col"]], s=5, alpha=0.4, color="darkorange")
axes[1].set_xlabel(CONFIG["rotor_speed_col"]); axes[1].set_ylabel(CONFIG["power_col"])
axes[1].set_title("Rotor Speed vs Power")

axes[2].scatter(ts[CONFIG["gen_temp_col"]], ts[CONFIG["bearing_temp_col"]], s=5, alpha=0.4, color="green")
axes[2].set_xlabel(CONFIG["gen_temp_col"]); axes[2].set_ylabel(CONFIG["bearing_temp_col"])
axes[2].set_title("Generator Temp vs Bearing Temp")

plt.tight_layout()
plt.show()


In [ ]:
### 3.2 Pair Plot ---------------------------------------------------------------
# Select a manageable subset of sensors — pair plots get unreadable with too many columns.

pairplot_cols = [
    CONFIG["power_col"], CONFIG["wind_speed_col"], CONFIG["rotor_speed_col"],
    CONFIG["gen_temp_col"], CONFIG["bearing_temp_col"],
]
pairplot_cols = [c for c in pairplot_cols if c in ts.columns]

sns.pairplot(ts[pairplot_cols].dropna().sample(min(2000, len(ts))), diag_kind="kde", plot_kws={"alpha": 0.4, "s": 10})
plt.suptitle("Pair Plot of Key Sensors", y=1.02)
plt.show()


In [ ]:
### 3.3 Correlation Heatmap ------------------------------------------------------

numeric_cols = ts.select_dtypes(include=[np.number]).columns
corr = ts[numeric_cols].corr()

plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=False, cmap="coolwarm", center=0, linewidths=0.3)
plt.title("Correlation Heatmap — All Numeric Sensors")
plt.tight_layout()
plt.show()


## 4. Time Series Analysis — Rolling Statistics (Part 3)

Covers: rolling mean, rolling std, rolling min/max, rolling median, moving-average smoothing.
Windows come from `CONFIG['rolling_windows']`.


In [ ]:
sensor_col = CONFIG["power_col"]  # change as needed

fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

axes[0].plot(ts.index, ts[sensor_col], alpha=0.3, label="Raw", linewidth=0.7)
for w in CONFIG["rolling_windows"]:
    axes[0].plot(ts.index, ts[sensor_col].rolling(w).mean(), label=f"Rolling Mean ({w})")
axes[0].set_title(f"Rolling Mean — {sensor_col}")
axes[0].legend()

for w in CONFIG["rolling_windows"]:
    axes[1].plot(ts.index, ts[sensor_col].rolling(w).std(), label=f"Rolling Std ({w})")
axes[1].set_title(f"Rolling Standard Deviation — {sensor_col}")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
### Rolling Min / Max / Median ---------------------------------------------------

w = CONFIG["rolling_windows"][-1]  # use the largest window for a clean view

plt.figure(figsize=(14, 5))
plt.plot(ts.index, ts[sensor_col], alpha=0.25, linewidth=0.7, label="Raw")
plt.plot(ts.index, ts[sensor_col].rolling(w).min(), label=f"Rolling Min ({w})")
plt.plot(ts.index, ts[sensor_col].rolling(w).max(), label=f"Rolling Max ({w})")
plt.plot(ts.index, ts[sensor_col].rolling(w).median(), label=f"Rolling Median ({w})", linewidth=2)
plt.title(f"Rolling Min / Max / Median — {sensor_col}")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
### Moving Average Smoothing (short-term fluctuation removal) --------------------

short_window = CONFIG["rolling_windows"][0]

plt.figure()
plt.plot(ts.index, ts[sensor_col], alpha=0.3, linewidth=0.6, label="Raw")
plt.plot(ts.index, ts[sensor_col].rolling(short_window, center=True).mean(), color="crimson", label=f"MA Smoothed ({short_window})")
plt.title(f"Moving Average Smoothing — {sensor_col}")
plt.legend()
plt.tight_layout()
plt.show()


## 5. Seasonality (Part 4)

Covers: seasonal decomposition, month-wise distribution, hour-of-day analysis.


In [ ]:
### 5.1 Seasonal Decomposition ----------------------------------------------------
# `period` should reflect your sampling frequency (e.g. 144 for 10-min data over a day, 24 for hourly).

series = ts[sensor_col].dropna()
period = 24  # <-- adjust to match your data's daily cycle length

if len(series) > 2 * period:
    decomposition = seasonal_decompose(series, model="additive", period=period)
    fig = decomposition.plot()
    fig.set_size_inches(14, 8)
    plt.tight_layout()
    plt.show()
else:
    print("Not enough data points for the chosen period — reduce `period` or provide more data.")


In [ ]:
### 5.2 Month-wise Distribution ----------------------------------------------------

ts_month = ts.copy()
ts_month["Month"] = ts_month.index.month_name()

plt.figure(figsize=(14, 6))
sns.boxplot(data=ts_month, x="Month", y=sensor_col,
            order=["January","February","March","April","May","June",
                   "July","August","September","October","November","December"])
plt.title(f"Month-wise Distribution — {sensor_col}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
### 5.3 Hour-of-Day Analysis ---------------------------------------------------------

ts_hour = ts.copy()
ts_hour["Hour"] = ts_hour.index.hour

plt.figure(figsize=(14, 6))
sns.boxplot(data=ts_hour, x="Hour", y=sensor_col)
plt.title(f"Hour-of-Day Distribution — {sensor_col}")
plt.tight_layout()
plt.show()


## 6. Autocorrelation Analysis (Part 5)

Covers: lag plot, ACF, PACF.


In [ ]:
### 6.1 Lag Plot -----------------------------------------------------------------

fig, axes = plt.subplots(1, len(CONFIG["lag_list"]), figsize=(5*len(CONFIG["lag_list"]), 4))
for ax, lag in zip(axes, CONFIG["lag_list"]):
    lag_plot(ts[sensor_col].dropna(), lag=lag, ax=ax, alpha=0.3, s=5)
    ax.set_title(f"Lag = {lag}")
plt.suptitle(f"Lag Plots — {sensor_col}", y=1.03)
plt.tight_layout()
plt.show()


In [ ]:
### 6.2 Autocorrelation Function (ACF) ---------------------------------------------

fig, ax = plt.subplots(figsize=(14, 4))
plot_acf(ts[sensor_col].dropna(), lags=max(CONFIG["lag_list"]) + 10, ax=ax)
plt.title(f"ACF — {sensor_col}")
plt.tight_layout()
plt.show()


In [ ]:
### 6.3 Partial Autocorrelation (PACF) -----------------------------------------------

fig, ax = plt.subplots(figsize=(14, 4))
plot_pacf(ts[sensor_col].dropna(), lags=max(CONFIG["lag_list"]) + 10, ax=ax, method="ywm")
plt.title(f"PACF — {sensor_col}")
plt.tight_layout()
plt.show()


## 7. Multi-Turbine Analysis (Part 6)

Covers: overlay time series, turbine-wise box plot, daily mean comparison,
monthly trend comparison, fleet heatmap.

These use the full `df` (not the single-turbine `ts`), so `CONFIG['turbine_col']` must be set.


In [ ]:
assert CONFIG["turbine_col"] in df.columns, "Set CONFIG['turbine_col'] to run multi-turbine analysis."

turbine_col = CONFIG["turbine_col"]
sensor_col = CONFIG["power_col"]

full = df.set_index(CONFIG["timestamp_col"]).sort_index()


In [ ]:
### 7.1 Overlay Time Series -------------------------------------------------------

plt.figure(figsize=(16, 6))
for turbine, grp in full.groupby(turbine_col):
    plt.plot(grp.index, grp[sensor_col], label=str(turbine), alpha=0.7, linewidth=0.8)
plt.title(f"Overlay Time Series — {sensor_col} by Turbine")
plt.legend(loc="upper right", ncol=4, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
### 7.2 Turbine-wise Box Plot -------------------------------------------------------

plt.figure(figsize=(14, 6))
sns.boxplot(data=full.reset_index(), x=turbine_col, y=sensor_col)
plt.title(f"Turbine-wise Distribution — {sensor_col}")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
### 7.3 Daily Mean Comparison --------------------------------------------------------

daily_mean = full.groupby(turbine_col)[sensor_col].resample("D").mean().unstack(level=0)

plt.figure(figsize=(16, 6))
daily_mean.plot(ax=plt.gca(), linewidth=0.8)
plt.title(f"Daily Mean {sensor_col} per Turbine")
plt.ylabel(sensor_col)
plt.legend(loc="upper right", ncol=4, fontsize=8)
plt.tight_layout()
plt.show()

# Ranking of average performance
print("Average power ranking (highest to lowest):")
print(daily_mean.mean().sort_values(ascending=False))


In [ ]:
### 7.4 Monthly Trend Comparison -----------------------------------------------------

monthly_mean = full.groupby(turbine_col)[sensor_col].resample("M").mean().unstack(level=0)

plt.figure(figsize=(16, 6))
monthly_mean.plot(ax=plt.gca(), marker="o")
plt.title(f"Monthly Trend Comparison — {sensor_col}")
plt.ylabel(sensor_col)
plt.legend(loc="upper right", ncol=4, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
### 7.5 Fleet-wide Heatmap (Turbines x Days) -------------------------------------------

fleet_daily = full.groupby(turbine_col)[sensor_col].resample("D").mean().unstack(level=0).T
# rows = turbines, columns = days

plt.figure(figsize=(18, 8))
sns.heatmap(fleet_daily, cmap="viridis", cbar_kws={"label": sensor_col})
plt.title(f"Fleet-wide Heatmap — Daily Mean {sensor_col}")
plt.xlabel("Day")
plt.ylabel("Turbine")
plt.tight_layout()
plt.show()


## 8. CMS Analysis (Part 7)

Covers: vibration trend, temperature trend, temperature vs vibration relationship.


In [ ]:
### 8.1 Vibration Trend --------------------------------------------------------------

plt.figure()
plt.plot(ts.index, ts[CONFIG["vibration_col"]], linewidth=0.7, color="purple")
plt.plot(ts.index, ts[CONFIG["vibration_col"]].rolling(CONFIG["rolling_windows"][-1]).mean(),
         color="black", label="Rolling Mean")
plt.title("Vibration Trend")
plt.ylabel(CONFIG["vibration_col"])
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
### 8.2 Temperature Trends -------------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 5))
for col, label in [
    (CONFIG["bearing_temp_col"], "Bearing Temp"),
    (CONFIG["gearbox_temp_col"], "Gearbox Temp"),
    (CONFIG["gen_temp_col"], "Generator Temp"),
]:
    if col in ts.columns:
        ax.plot(ts.index, ts[col], label=label, linewidth=0.8)
ax.set_title("Temperature Trends")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
### 8.3 Temperature vs Vibration --------------------------------------------------------

plt.figure()
plt.scatter(ts[CONFIG["bearing_temp_col"]], ts[CONFIG["vibration_col"]], s=6, alpha=0.4)
plt.xlabel(CONFIG["bearing_temp_col"])
plt.ylabel(CONFIG["vibration_col"])
plt.title("Bearing Temperature vs Vibration")
plt.tight_layout()
plt.show()


## 9. Feature Engineering Visualization (Part 8)

Covers: lag features, rate of change, percentage change, rolling statistics recap,
cumulative sum, and before/after scaling comparison.


In [ ]:
### 9.1 Lag Features -------------------------------------------------------------------

feat = ts[[sensor_col]].copy()
for lag in CONFIG["lag_list"]:
    feat[f"lag_{lag}"] = feat[sensor_col].shift(lag)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(feat.index, feat[sensor_col], label="Original", linewidth=0.8)
for lag in CONFIG["lag_list"]:
    ax.plot(feat.index, feat[f"lag_{lag}"], label=f"Lag {lag}", alpha=0.6, linewidth=0.8)
ax.set_title(f"Lag Features — {sensor_col}")
ax.legend()
plt.tight_layout()
plt.show()

feat.head(10)


In [ ]:
### 9.2 Rate of Change (ROC) -------------------------------------------------------------

roc = ts[sensor_col].diff(CONFIG["roc_lag"])

plt.figure()
plt.plot(ts.index, roc, linewidth=0.7, color="firebrick")
plt.axhline(0, color="black", linewidth=0.8)
plt.title(f"Rate of Change — {sensor_col} (lag={CONFIG['roc_lag']})")
plt.tight_layout()
plt.show()


In [ ]:
### 9.3 Percentage Change ------------------------------------------------------------------

pct = ts[sensor_col].pct_change(CONFIG["roc_lag"]) * 100

plt.figure()
plt.plot(ts.index, pct, linewidth=0.7, color="teal")
plt.axhline(0, color="black", linewidth=0.8)
plt.title(f"Percentage Change (%) — {sensor_col}")
plt.ylabel("% change")
plt.tight_layout()
plt.show()


In [ ]:
### 9.4 Rolling Statistics Recap (all-in-one view) -----------------------------------------

w = CONFIG["rolling_windows"][-1]
roll_df = pd.DataFrame({
    "raw": ts[sensor_col],
    "rolling_mean": ts[sensor_col].rolling(w).mean(),
    "rolling_std": ts[sensor_col].rolling(w).std(),
    "rolling_min": ts[sensor_col].rolling(w).min(),
    "rolling_max": ts[sensor_col].rolling(w).max(),
    "rolling_median": ts[sensor_col].rolling(w).median(),
})

roll_df[["raw", "rolling_mean", "rolling_min", "rolling_max", "rolling_median"]].plot(figsize=(14, 6), alpha=0.8)
plt.title(f"Rolling Statistics Recap — {sensor_col} (window={w})")
plt.tight_layout()
plt.show()


In [ ]:
### 9.5 Cumulative Sum ----------------------------------------------------------------------

cumsum = ts[sensor_col].fillna(0).cumsum()

plt.figure()
plt.plot(ts.index, cumsum, color="darkgreen")
plt.title(f"Cumulative Sum — {sensor_col}")
plt.ylabel(f"Cumulative {sensor_col}")
plt.tight_layout()
plt.show()


In [ ]:
### 9.6 Feature Distribution: Before vs After Scaling --------------------------------------

values = ts[[sensor_col]].dropna()

standard_scaled = StandardScaler().fit_transform(values)
minmax_scaled = MinMaxScaler().fit_transform(values)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(values[sensor_col], bins=40, ax=axes[0], kde=True)
axes[0].set_title("Original")

sns.histplot(standard_scaled.flatten(), bins=40, ax=axes[1], kde=True, color="orange")
axes[1].set_title("StandardScaler")

sns.histplot(minmax_scaled.flatten(), bins=40, ax=axes[2], kde=True, color="green")
axes[2].set_title("MinMaxScaler")

plt.suptitle(f"Feature Distribution Before & After Scaling — {sensor_col}", y=1.03)
plt.tight_layout()
plt.show()


## 10. Pre-Modeling Checklist (Part 9)

Quick automated checks before moving to Feature Engineering / Modeling
(Isolation Forest, Autoencoder, LSTM, GRU, Transformer, etc.).


In [ ]:
print("=" * 60)
print("PRE-MODELING CHECKLIST")
print("=" * 60)

# 1. Missing values
print("\n[Missing Values]")
print(ts.isna().sum()[ts.isna().sum() > 0])

# 2. Duplicate records
print(f"\n[Duplicate Rows] {ts.duplicated().sum()}")

# 3. Constant features
constant_cols = [c for c in ts.select_dtypes(include=[np.number]).columns if ts[c].nunique() <= 1]
print(f"\n[Constant Features] {constant_cols if constant_cols else 'None found'}")

# 4. Outliers (IQR method) for the focus sensor
q1, q3 = ts[sensor_col].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = ts[(ts[sensor_col] < lower) | (ts[sensor_col] > upper)]
print(f"\n[Outliers in {sensor_col} — IQR method] {len(outliers)} rows ({len(outliers)/len(ts):.2%})")

# 5. Basic sensor drift check: compare first vs last chunk mean
n = len(ts) // 10 or 1
drift = ts[sensor_col].iloc[-n:].mean() - ts[sensor_col].iloc[:n].mean()
print(f"\n[Sensor Drift] mean(last 10%) - mean(first 10%) = {drift:.3f}")

print("\nReview the plots above for: Correlation, Seasonality, Trend, Feature Distribution.")
print("Once satisfied, proceed to Feature Scaling, Lag/Rolling Features, and a time-based Train/Test split.")


In [ ]:
### Time-based Train/Test Split (example) --------------------------------------------------

split_ratio = 0.8
split_idx = int(len(ts) * split_ratio)

train = ts.iloc[:split_idx]
test = ts.iloc[split_idx:]

print(f"Train: {train.index.min()} -> {train.index.max()}  ({len(train)} rows)")
print(f"Test:  {test.index.min()} -> {test.index.max()}  ({len(test)} rows)")


---
### Next Steps

With this EDA complete, proceed to **Feature Engineering** (lag features, rolling
statistics, rate-of-change, scaling) using the time-based train/test split above, then
move on to modeling (Isolation Forest, Autoencoder, LSTM, GRU, or Transformer) for
anomaly detection / RUL / degradation prediction.
